# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start(

1) Load and investigate the data

In [2]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [ ]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

# def new_fingerprint_function here

fpgens = {
    "MorganFP": morganfp,
    "MACCSkeys": maccskeys
    # add new fp here
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MACCSkeys
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x11cf5e7a0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x11cf5e810>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x11cf5e880>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x11cf5e8f0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x11cf5e960>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [4]:
X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [ ]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(2048, 256)   # input layer with 2048 features and output layer with 256 neurons
        #self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(256, 64)    # hidden layer with 256 neurons and output layer with 64 neurons
        #self.dropout = nn.Dropout(p=0.5)
        self.fc3 = nn.Linear(64, 1)      # output layer with 64 neurons and single output neuron
        #self.sigmoid = nn.Sigmoid()

        self.dropout = nn.Dropout(p=0.5)  # 50% der Neuronen werden zufällig deaktiviert
        #dropout has to be included between every layer!!
        #deactivate last one

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x) # BCEWithLogitsLoss() includes the sigmoid activation, so we don't apply it here.

        return x


In [6]:
# Parameters (change and add as needed)
learning_rate = 0.001
num_epochs = 50

In [7]:
model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCEWithLogitsLoss() # Binary Cross Entropy Loss

# choose an optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [ ]:
lowest_test_loss = float('inf')
lowest_test_loss_epoch = 0
best_morganfp_model_state = None  # Variable to store the best model state

# running the training in batches increses accuracy significantly! (but computationally a bit more expensive)

for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train_tensor).squeeze()
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    model.eval()
    with torch.no_grad():
        test_loss = criterion(model(X_test_tensor).squeeze(), y_test_tensor)
        if test_loss.item() < lowest_test_loss:
            lowest_test_loss = test_loss.item()
            lowest_test_loss_epoch = epoch
            best_morganfp_model_state = model.state_dict()  # save the best model state

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

print(f'Lowest Test Loss: {lowest_test_loss:.4f} at epoch {lowest_test_loss_epoch + 1}')
model.load_state_dict(best_morganfp_model_state)  # load the best model state for further evaluation or inference

Epoch [10/50], Train Loss: 0.6222, Test Loss: 0.6142
Epoch [20/50], Train Loss: 0.4646, Test Loss: 0.5006
Epoch [30/50], Train Loss: 0.3602, Test Loss: 0.4901
Epoch [40/50], Train Loss: 0.2848, Test Loss: 0.5062
Epoch [50/50], Train Loss: 0.2331, Test Loss: 0.5276
Lowest Test Loss: 0.4870 at epoch 26


<All keys matched successfully>

6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [9]:
y_pred_prob = torch.sigmoid(model(X_test_tensor)).squeeze().detach().tolist()
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int) # convert probabilities to binary predictions using a threshold of 0.5

# evaluate the model using accuracy, test-ROC-AUC
from sklearn.metrics import accuracy_score, roc_auc_score
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f'Accuracy: {accuracy:.4f}')
print(f'Test ROC-AUC: {roc_auc:.4f}')


Accuracy: 0.7926
Test ROC-AUC: 0.8596


MD: Repeat for MACCSkeys

In [10]:
X = np.stack(df["MACCSkeys"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        self.fc1 = nn.Linear(167, 84)
        self.fc2 = nn.Linear(84, 42)
        self.fc3 = nn.Linear(42, 21)
        self.fc4 = nn.Linear(21, 1)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

learning_rate = 0.002
num_epochs = 200

model = BinClassifierNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

lowest_test_loss = float('inf')
lowest_test_loss_epoch = 0
best_maccskeys_model_state = None

for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train_tensor).squeeze()
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_loss = criterion(model(X_test_tensor).squeeze(), y_test_tensor)
        if test_loss.item() < lowest_test_loss:
            lowest_test_loss = test_loss.item()
            lowest_test_loss_epoch = epoch
            best_maccskeys_model_state = model.state_dict()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

print(f'Lowest Test Loss: {lowest_test_loss:.4f} at epoch {lowest_test_loss_epoch + 1}')
model.load_state_dict(best_maccskeys_model_state)

y_pred_prob = torch.sigmoid(model(X_test_tensor)).squeeze().detach().tolist()
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f'Accuracy: {accuracy:.4f}')
print(f'Test ROC-AUC: {roc_auc:.4f}')


Epoch [10/200], Train Loss: 0.6439, Test Loss: 0.6297
Epoch [20/200], Train Loss: 0.5862, Test Loss: 0.5708
Epoch [30/200], Train Loss: 0.5415, Test Loss: 0.5248
Epoch [40/200], Train Loss: 0.5002, Test Loss: 0.4963
Epoch [50/200], Train Loss: 0.4652, Test Loss: 0.4732
Epoch [60/200], Train Loss: 0.4297, Test Loss: 0.4616
Epoch [70/200], Train Loss: 0.4067, Test Loss: 0.4549
Epoch [80/200], Train Loss: 0.3765, Test Loss: 0.4516
Epoch [90/200], Train Loss: 0.3543, Test Loss: 0.4471
Epoch [100/200], Train Loss: 0.3300, Test Loss: 0.4503
Epoch [110/200], Train Loss: 0.3110, Test Loss: 0.4544
Epoch [120/200], Train Loss: 0.2931, Test Loss: 0.4599
Epoch [130/200], Train Loss: 0.2708, Test Loss: 0.4544
Epoch [140/200], Train Loss: 0.2479, Test Loss: 0.4619
Epoch [150/200], Train Loss: 0.2446, Test Loss: 0.4690
Epoch [160/200], Train Loss: 0.2286, Test Loss: 0.4761
Epoch [170/200], Train Loss: 0.2178, Test Loss: 0.4875
Epoch [180/200], Train Loss: 0.2081, Test Loss: 0.4932
Epoch [190/200], Tr

7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [11]:
# two options to save a pytorch model:

# option 1 (recommended): save only the weights (state_dict)
# more portable - only depends on the model architecture, not the python/pytorch version
torch.save(best_morganfp_model_state, 'model_morganfp.pth')
torch.save(best_maccskeys_model_state, 'model_maccs.pth')

# option 2: save the full model
# not recommended - tied to the exact python/pytorch version and class structure
torch.save(model, 'model_maccs_full.pth')

# loading option 1 - architecture must be defined first
model = BinClassifierNN()
model.load_state_dict(torch.load('model_maccs.pth'))

# loading option 2
model = torch.load('model_maccs_full.pth')

#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?


1. the nn performed similarly to the baseline models. accuracy 0.79 and roc-auc 0.86 is on par with knn, and not far off random forest (0.83 / 0.90) which was the best baseline. not bad for a simple nn, but no clear advantage over classical ml here.

2. yes, maccs keys performed slightly better despite being much smaller (167 vs 2048 bits). maccs keys seems to give the nn a better signal to noise ratio than the sparse morgan fp. furthermore, maccskeys were significantly less prone to overfitting.

3. bigger isn't always better, which not only holds for fingerprints but also for the nn itslef. morgan fp has 2048 bits but most are zero for any given molecule, which makes it harder to learn from. more features also means more risk of overfitting. maccs keys are smaller but more information-dense.

4. for morgan fp: 2048→256→64→1 neurons, dropout 0.5, lr 0.001, 26 epochs. for maccs keys: 167→84→42→21→1 neurons, dropout 0.3, lr 0.002, 100-120 epochs optimal.
To some respect it pays off to optimize, but with small data sets there are limitations, where further finetuning does not bring significantly better results.

5. yes, especially for morgan fp. train loss went to 0.0008 while test loss kept rising to more than 2. used a high dropout, small learning rate and early stopping (saving best model state), when test loss was minimal. other options would be l2 regularization (weight decay in adam), more training data, or batch normalization.
similarly to RF, the model will find a way to reach 100% accuracy for the test data.

6. noise here could come from experimental variability (false poitives / negatives). the same compound tested in different labs or assays can give different results. the model could be used in early drug discovery to filter out potentially mutagenic compounds before synthesis. other useful tools in this qsar context would be graph neural networks (gnns) which work directly on molecular graphs, or attention-based models that could highlight which substructures drive mutagenicity.

7. the full model is tied to the exact pytorch version and class definition. if one of them changes, loading fails. state_dict only saves the weights, so it's more portable and safer for long-term storage.
besides that: size and redundancy